In [1]:
import re
import pandas as pd

CODEBOOK = "Codebook_HSLS_17_PETS.txt"

In [2]:
text = open(CODEBOOK, encoding="utf-8-sig").read()
entries = re.split(r"-{80,}", text)
print(f"Number of blocks: {len(entries)}")

Number of blocks: 20603


In [5]:
# Find a block containing a variable you know well
for i, entry in enumerate(entries):
    if "Name:       X1SES" in entry:
        print(f"Block index: {i}")
        print(entry[:1500])
        break

Block index: 210



File:       STUDENT
Name:       X1SES
Position:   587
Length:     7
Label:      X1 Socio-economic status composite

Description:
This composite variable is used to measure a construct for socioeconomic status. X1SES is calculated using parent/guardians' education (X1PAR1EDU and X1PAR2EDU), occupation (X1PAR1OCC2 and X1PAR2OCC2), and family income (X1FAMINCOME). For cases with nonresponding parent/guardians, 5 imputed values are generated (X1SES1-X1SES5), X1SES is computed as the average of the 5 imputed values, and the imputation flag is set as X1SES_IM=1 (values for parent/guardian education, occupation, and income are set to -8). When education, occupation, or family income are imputed using other information provided by the responding parent/guardian, X1SES is constructed from the combination of actual and imputed parent/guardian values. For these cases, the values of X1SES1-X1SES5 are equivalent to X1SES and X1SES_IM=2. Otherwise, the responding parent/guardian 

In [6]:
for i, entry in enumerate(entries):
    if "Name:       X3DROPSTAT" in entry:
        print(entry[:2000])
        break




File:       STUDENT
Name:       X3DROPSTAT
Position:   1352
Length:     2
Label:      X3 U13 dropout status

Description:
Sample member status of dropout from high school.


SAS Logic:
if X3SQSTAT=8 then X3DROPSTAT=-8;
else if X3HSCRED=1 and X3HSCREDTYPE=1 then X3DROPSTAT=0;
else if X3HSCRED=0 and ((201300<X3LASTHSDATE<201304) or 0<X3LASTHSDATE<201300) then X3DROPSTAT=1;
else if X3HSCRED=1 and X3HSCREDTYPE in (2,3) then X3DROPSTAT=2;
else X3DROPSTAT=4;
if X3DROPSTAT in (0,4) and X2EVERDROP=1 then X3DROPSTAT=3;
Sources: 2013 update questionnaire

                                                                        Frequency             Percent 
Category            Label                                              Unweighted          Unweighted 
------------------- ----------------------------------------- ------------------- ------------------- 
0                   Not dropout/alternative completer                      15,535               66.10 
1                   Spring term 2

In [7]:
def parse_entry(entry):
    name_m    = re.search(r"Name:\s+(\S+)", entry)
    label_m   = re.search(r"Label:\s+(.+)", entry)
    desc_m    = re.search(r"Description:\s*\n(.*?)(?=SAS Logic:|Sources:|Category\s+Label|Mean\s+Std|\Z)", entry, re.DOTALL)
    sas_m     = re.search(r"SAS Logic:\s*\n(.*?)(?=Sources:|Category\s+Label|Mean\s+Std|\Z)", entry, re.DOTALL)
    src_m     = re.search(r"Sources?:\s*(.+?)(?:\n\n|\Z)", entry, re.DOTALL)

    if not name_m:
        return None

    return {
        "variable":    name_m.group(1).strip(),
        "label":       label_m.group(1).strip() if label_m else "",
        "description": desc_m.group(1).strip()  if desc_m  else "",
        "sas_logic":   sas_m.group(1).strip()   if sas_m   else "",
        "sources":     src_m.group(1).strip()    if src_m   else "",
    }

parsed = [parse_entry(e) for e in entries]
parsed = [p for p in parsed if p is not None]
print(f"Successfully parsed: {len(parsed)}")

Successfully parsed: 10301


In [8]:
# Check a description composite
x1ses = next(p for p in parsed if p["variable"] == "X1SES")
print("=== X1SES ===")
print(f"Label:       {x1ses['label']}")
print(f"Description: {x1ses['description'][:200]}")
print(f"SAS Logic:   '{x1ses['sas_logic']}'")
print(f"Sources:     {x1ses['sources']}")

print()

# Check a SAS composite
x3drop = next(p for p in parsed if p["variable"] == "X3DROPSTAT")
print("=== X3DROPSTAT ===")
print(f"Label:       {x3drop['label']}")
print(f"Description: {x3drop['description'][:200]}")
print(f"SAS Logic:   {x3drop['sas_logic'][:200]}")
print(f"Sources:     {x3drop['sources']}")

=== X1SES ===
Label:       X1 Socio-economic status composite
Description: This composite variable is used to measure a construct for socioeconomic status. X1SES is calculated using parent/guardians' education (X1PAR1EDU and X1PAR2EDU), occupation (X1PAR1OCC2 and X1PAR2OCC2)
SAS Logic:   ''
Sources:     

=== X3DROPSTAT ===
Label:       X3 U13 dropout status
Description: Sample member status of dropout from high school.
SAS Logic:   if X3SQSTAT=8 then X3DROPSTAT=-8;
else if X3HSCRED=1 and X3HSCREDTYPE=1 then X3DROPSTAT=0;
else if X3HSCRED=0 and ((201300<X3LASTHSDATE<201304) or 0<X3LASTHSDATE<201300) then X3DROPSTAT=1;
else if X3H
Sources:     2013 update questionnaire


In [9]:
col_names = open("hsls_column_names.txt").read().strip().split("\n")

parsed_names = {p["variable"] for p in parsed}
missing = [v for v in col_names if v not in parsed_names]

print(f"Your variables:     {len(col_names)}")
print(f"Found in codebook:  {len(col_names) - len(missing)}")
print(f"Missing:            {len(missing)}")
print(missing)

Your variables:     913
Found in codebook:  913
Missing:            0
[]


In [10]:
codebook_df = pd.DataFrame(parsed)
codebook_df = codebook_df[codebook_df["variable"].isin(col_names)].reset_index(drop=True)
print(codebook_df.shape)
codebook_df.head()

(1125, 5)


,variable,label,description,sas_logic,sources
0,STU_ID,Student ID,Student identifier assigned for all base year ...,,
1,X1SEX,X1 Student's sex,"Sex of the sample member, taken from the base ...",,
2,X1RACE,X1 Student's race/ethnicity-composite,X1RACE characterizes the sample member's race/...,,
3,X1HISPANIC,X1 Student is Hispanic/Latino/Latina-composite,The sample member's race/ethnicity is characte...,,
4,X1WHITE,X1 Student is White-composite,The sample member's race/ethnicity is characte...,,


In [11]:
dups = codebook_df[codebook_df.duplicated(subset="variable", keep=False)]
print(f"Duplicate variable names: {dups['variable'].nunique()}")
print(dups["variable"].value_counts().head(20))

Duplicate variable names: 212
variable
X1CONTROL       2
X1LOCALE        2
X1REGION        2
X1SCHOOLCLI     2
X1COUPERTEA     2
X1COUPERCOU     2
X1COUPERPRI     2
X1AQSTAT        2
X1AQDATE        2
X1AQDESIGNEE    2
X1CQSTAT        2
X1CQDATE        2
A1SCHCONTROL    2
A1NOTIFY        2
A1MTHSCIFAIR    2
A1MSSUMMER      2
A1MSAFTERSCH    2
A1MSMENTOR      2
A1MSSPEAKER     2
A1MSFLDTRIP     2
Name: count, dtype: int64


In [12]:
codebook_df = codebook_df.drop_duplicates(subset="variable", keep="first").reset_index(drop=True)
print(codebook_df.shape)

(913, 5)


In [13]:
def get_instrument(v):
    if v.startswith("X1"):   return "NCES_composite"
    if v.startswith("S1"):   return "student_BY"
    if v.startswith("P1"):   return "parent_BY"
    if v.startswith("M1"):   return "mathteacher_BY"
    if v.startswith("A1"):   return "admin_BY"
    if v.startswith("C1"):   return "counselor_BY"
    if v == "STU_ID":        return "identifier"
    if v == "X4EVERDROP":    return "target"
    return "other"

codebook_df["instrument"] = codebook_df["variable"].apply(get_instrument)
codebook_df["instrument"].value_counts()

instrument
student_BY        271
NCES_composite    161
counselor_BY      150
parent_BY         141
mathteacher_BY    134
admin_BY           54
identifier          1
target              1
Name: count, dtype: int64

In [14]:
def get_construct_type(row):
    if row["sas_logic"]:
        return "sas_composite"
    if any(kw in row["description"].lower() for kw in [
        "calculated using", "inputs to", "constructed from",
        "derived from", "based on", "computed from",
        "created from", "uses ", "using "
    ]):
        return "description_composite"
    return "raw"

codebook_df["construct_type"] = codebook_df.apply(get_construct_type, axis=1)
codebook_df["construct_type"].value_counts()

construct_type
raw                      704
description_composite    208
sas_composite              1
Name: count, dtype: int64

In [15]:
# Tokens to ignore when scanning for variable name references
IGNORE_TOKENS = {
    "IF", "THEN", "ELSE", "AND", "OR", "NOT", "IN", "DO", "END",
    "NULL", "MISSING", "MAX", "MIN", "SUM", "MEAN", "ABS", "INT",
    "SUBSTR", "UPCASE", "PUT", "INPUT", "CAT", "CATS", "CATX",
    "LENGTH", "LABEL", "FORMAT", "INFORMAT", "RETAIN", "ARRAY",
    "OUTPUT", "RETURN", "STOP", "RUN", "DATA", "SET", "MERGE",
    "BY", "WHERE", "KEEP", "DROP", "RENAME", "CLASS", "MODEL",
    "VAR", "WEIGHT", "FREQ", "TABLES", "TRUE", "FALSE",
    "NCESID", "CCD", "PSS", "NCES", "GED", "IEP", "AP", "IB",
    "ONET", "STEM", "SAT", "ACT", "PSAT", "GPA",
}

VARNAME_RE = re.compile(r'\b([A-Z][A-Z0-9_]{2,})\b')
all_known  = set(codebook_df["variable"])

def extract_parents(row):
    text = row["sas_logic"] if row["sas_logic"] else row["description"]
    candidates = VARNAME_RE.findall(text)
    parents = set()
    for c in candidates:
        if c == row["variable"]:      continue
        if c in IGNORE_TOKENS:        continue
        if c not in all_known:        continue
        if not any(ch.isdigit() for ch in c): continue
        parents.add(c)
    return sorted(parents)

codebook_df["parents"] = codebook_df.apply(extract_parents, axis=1)
codebook_df["n_parents"] = codebook_df["parents"].apply(len)

# Verify on known variables
for v in ["X1SES", "X1MTHID", "X1SCHOOLBEL", "X3DROPSTAT"]:
    if v in all_known:
        row = codebook_df[codebook_df["variable"] == v].iloc[0]
        print(f"{v}: {row['parents']}")

X1SES: ['X1FAMINCOME', 'X1PAR1EDU', 'X1PAR1OCC2', 'X1PAR2EDU', 'X1PAR2OCC2', 'X1SES1', 'X1SES5', 'X1SES_IM']
X1MTHID: ['S1MPERSON1', 'S1MPERSON2']
X1SCHOOLBEL: ['S1GOODGRADES', 'S1PROUD', 'S1SAFE', 'S1SCHWASTE', 'S1TALKPROB']


In [16]:
def compute_depth_and_leaves(varname, memo, visiting):
    if varname in memo:
        return memo[varname]
    if varname in visiting:
        # cycle detected - break it
        return (0, {varname})
    
    visiting.add(varname)
    
    row = codebook_df[codebook_df["variable"] == varname]
    if row.empty:
        result = (0, {varname})
    else:
        parents = row.iloc[0]["parents"]
        if not parents:
            result = (0, {varname})
        else:
            child_results = [compute_depth_and_leaves(p, memo, visiting) for p in parents]
            max_depth = max(r[0] for r in child_results)
            all_leaves = set().union(*[r[1] for r in child_results])
            result = (max_depth + 1, all_leaves)
    
    visiting.remove(varname)
    memo[varname] = result
    return result

memo = {}
for v in codebook_df["variable"]:
    compute_depth_and_leaves(v, memo, set())

codebook_df["depth"]     = codebook_df["variable"].apply(lambda v: memo[v][0])
codebook_df["n_leaves"]  = codebook_df["variable"].apply(lambda v: len(memo[v][1]))

# Verify
for v in ["X1SES", "X1MTHID", "X1SCHOOLBEL", "S1GOODGRADES"]:
    if v in all_known:
        row = codebook_df[codebook_df["variable"] == v].iloc[0]
        print(f"{v}: depth={row['depth']}, n_leaves={row['n_leaves']}")

X1SES: depth=5, n_leaves=16
X1MTHID: depth=1, n_leaves=2
X1SCHOOLBEL: depth=1, n_leaves=5
S1GOODGRADES: depth=0, n_leaves=1


In [17]:
codebook_df.to_csv("codebook_parsed.csv", index=False)
print(codebook_df.shape)
codebook_df.head()

(913, 11)


,variable,label,description,sas_logic,sources,instrument,construct_type,parents,n_parents,depth,n_leaves
0,STU_ID,Student ID,Student identifier assigned for all base year ...,,,identifier,raw,[],0,0,1
1,X1SEX,X1 Student's sex,"Sex of the sample member, taken from the base ...",,,NCES_composite,description_composite,[],0,0,1
2,X1RACE,X1 Student's race/ethnicity-composite,X1RACE characterizes the sample member's race/...,,,NCES_composite,description_composite,"[X1BLACK, X1HISPANIC, X1WHITE]",3,4,3
3,X1HISPANIC,X1 Student is Hispanic/Latino/Latina-composite,The sample member's race/ethnicity is characte...,,,NCES_composite,description_composite,"[X1BLACK, X1RACE, X1WHITE]",3,2,3
4,X1WHITE,X1 Student is White-composite,The sample member's race/ethnicity is characte...,,,NCES_composite,description_composite,"[X1BLACK, X1HISPANIC, X1RACE]",3,1,3


In [39]:
import json
import os

SAVE_FILE = "tier1_assignments.json"

# Load existing progress if it exists
if os.path.exists(SAVE_FILE):
    assignments = json.load(open(SAVE_FILE))
else:
    assignments = {}

for _, row in codebook_df.iterrows():
    v = row["variable"]
    
    # Skip identifier and target
    if row["instrument"] in ("identifier", "target"):
        continue
    
    # Skip already assigned
    if v in assignments:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable:    {v}")
    print(f"Label:       {row['label']}")
    print(f"Instrument:  {row['instrument']}")
    print(f"Description: {row['description'][:400]}")
    
    tier1 = input("\nTier 1? (y/n/q to quit): ").strip().lower()
    if tier1 == "q":
        break
    
    certain = input("Certain? (y/n): ").strip().lower()
    
    assignments[v] = {
        "tier_1":   tier1 == "y",
        "certain":  certain == "y",
    }
    
    json.dump(assignments, open(SAVE_FILE, "w"), indent=2)

print(f"\nProgress: {len(assignments)}/911 variables assigned")



Variable:    X1TXMTH1
Label:       X1 Mathematics theta score - multiple imputation value 1 of 5
Instrument:  NCES_composite
Description: Mathematics theta score multiple imputation value (1 of 5). When the math test data were missing for student survey respondents, the math theta score was imputed with multiple imputation technique, with 5 imputed values. X1TXMTH is the mean of X1TXMTH1-5.  The theta score provides a norm-referenced measurement of achievement, that is, an estimate of achievement relative to the population (fall 200

Variable:    X1TXMTH2
Label:       X1 Mathematics theta score - multiple imputation value 2 of 5
Instrument:  NCES_composite
Description: Mathematics theta score multiple imputation value (2 of 5). When the math test data were missing for student survey respondents, the math theta score was imputed with multiple imputation technique, with 5 imputed values. X1TXMTH is the mean of X1TXMTH1-5.  The theta score provides a norm-referenced measurement of achiev

In [37]:
assignments = json.load(open("tier1_assignments.json"))

for variable, decision in assignments.items():
    if decision["certain"]:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable: {variable}")
    print(f"Current:  tier_1={decision['tier_1']}")
    
    # Show label from codebook_df
    row = codebook_df[codebook_df["variable"] == variable]
    if not row.empty:
        print(f"Label:    {row.iloc[0]['label']}")
        print(f"Description: {row.iloc[0]['description'][:300]}")
    
    tier1 = input("\nTier 1? (y/n/q to quit): ").strip().lower()
    if tier1 == "q":
        break
    
    assignments[variable]["tier_1"] = (tier1 == "y")
    assignments[variable]["certain"] = True
    
    json.dump(assignments, open("tier1_assignments.json", "w"), indent=2)

print("Done")


Variable: C1PLANPARENT
Current:  tier_1=True
Label:    C1 A14 School shares students' career/education plans with their parents
Description: Does your school share students' [career and education/education/career] plans with their parents or guardians?
  Yes
  No

Note:  Question wording was customized in the survey instrument based on whether the respondent previously indicated their school required students to have a combined career an

Variable: C1SIGNOFF
Current:  tier_1=True
Label:    C1 A15 School requires parents to sign off on students' career/education plans
Description: Are parents or guardians required to sign off on students' [career and education/education/career] plans?
  Yes
  No

Note:  Question wording was customized in the survey instrument based on whether the respondent previously indicated their school required students to have a combined career and educ

Variable: C1TECHSUPPRT
Current:  tier_1=False
Label:    C1 B16A School supports students with technology/softw